# Notebook 2 — ELT Pipeline Olist + Brazil Holiday API

Notebook ini dibuat untuk **Bagian II – Pipeline ELT** pada Tugas Besar Big Data.

Perbedaan utama dengan ETL:

- **ETL**: data diekstrak, dibersihkan/ditransformasi dengan Python/Pandas terlebih dahulu, lalu dimuat ke warehouse.
- **ELT**: data mentah diekstrak dan **langsung dimuat ke raw tables di warehouse**, lalu transformasi dilakukan **setelah data berada di warehouse** menggunakan SQL.

Sumber data yang dipakai sama dengan ETL:

1. Source 1: Olist CSV dari Kaggle.
2. Source 2: Nager.Date Public Holiday API Brazil 2016–2018.

Output utama notebook ini:

- SQLite raw warehouse: `bigdata_warehouse_elt.db`
- Raw tables: `raw_customers`, `raw_orders`, `raw_order_items`, dll.
- Staging tables: `stg_payments_agg`, `stg_reviews_agg`, `stg_geolocation_agg`, `stg_products_en`, `stg_fact_order_items_enriched`
- Star schema ELT: `fact_order_items_elt`, `dim_customer_elt`, `dim_product_elt`, `dim_seller_elt`, `dim_payment_elt`, `dim_date_elt`
- Metadata load log dan validation results.


## 0. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Import Library dan Setup Path

In [ ]:
import os
import time
import json
import sqlite3
import requests
import numpy as np
import pandas as pd
from datetime import datetime

PROJECT_PATH = '/content/drive/MyDrive/bigdata_final_project'
RAW_OLIST_PATH = f'{PROJECT_PATH}/raw/olist'
RAW_HOLIDAY_PATH = f'{PROJECT_PATH}/raw/holidays'
WAREHOUSE_PATH = f'{PROJECT_PATH}/warehouse'
LOG_PATH = f'{PROJECT_PATH}/elt_pipeline/logs'

for path in [RAW_HOLIDAY_PATH, WAREHOUSE_PATH, LOG_PATH]:
    os.makedirs(path, exist_ok=True)

print('Project path siap:', PROJECT_PATH)

Project path siap: /content/drive/MyDrive/bigdata_final_project


## 2. Fungsi Logging dan Metadata ELT

In [ ]:
elt_logs = []
load_metadata = []

def get_file_size_mb(file_path):
    if os.path.exists(file_path):
        return round(os.path.getsize(file_path) / (1024 * 1024), 3)
    return None


def add_elt_log(step, object_name, rows=None, cols=None, execution_time_sec=None, status='success', notes=''):
    elt_logs.append({
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'step': step,
        'object_name': object_name,
        'rows': rows,
        'cols': cols,
        'execution_time_sec': execution_time_sec,
        'status': status,
        'notes': notes
    })


def add_metadata(source_name, target_table, rows, cols, file_size_mb=None, notes=''):
    load_metadata.append({
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'source_name': source_name,
        'target_table': target_table,
        'rows': rows,
        'cols': cols,
        'file_size_mb': file_size_mb,
        'notes': notes
    })


def save_elt_logs():
    log_df = pd.DataFrame(elt_logs)
    metadata_df = pd.DataFrame(load_metadata)

    log_file = f'{LOG_PATH}/elt_process_log.csv'
    metadata_file = f'{LOG_PATH}/elt_load_metadata.csv'

    log_df.to_csv(log_file, index=False)
    metadata_df.to_csv(metadata_file, index=False)

    print('Log ELT tersimpan di:', log_file)
    print('Metadata load ELT tersimpan di:', metadata_file)

    return log_df, metadata_df

## 3. Extract Source 1 — Baca Raw CSV Olist Tanpa Preprocessing

In [ ]:
def extract_elt_source1():
    start_time = time.time()

    file_map = {
        'customers': 'olist_customers_dataset.csv',
        'geolocation': 'olist_geolocation_dataset.csv',
        'order_items': 'olist_order_items_dataset.csv',
        'payments': 'olist_order_payments_dataset.csv',
        'reviews': 'olist_order_reviews_dataset.csv',
        'orders': 'olist_orders_dataset.csv',
        'products': 'olist_products_dataset.csv',
        'sellers': 'olist_sellers_dataset.csv',
        'translation': 'product_category_name_translation.csv'
    }

    data = {}
    for name, file_name in file_map.items():
        file_path = f'{RAW_OLIST_PATH}/{file_name}'
        df = pd.read_csv(file_path)
        data[name] = df

        add_elt_log(
            step='extract',
            object_name=f'raw_csv_{name}',
            rows=df.shape[0],
            cols=df.shape[1],
            notes=f'Extract raw CSV from {file_path}; no preprocessing applied'
        )

    execution_time = round(time.time() - start_time, 3)
    add_elt_log('extract_summary', 'Olist CSV raw extraction', execution_time_sec=execution_time)
    return data

olist_raw = extract_elt_source1()
print('Extract source 1 selesai.')

Extract source 1 selesai.


## 4. Extract Source 2 — Holiday API Tanpa Preprocessing

In [ ]:
def extract_elt_source2():
    start_time = time.time()
    years = [2016, 2017, 2018]
    records = []

    for year in years:
        url = f'https://date.nager.at/api/v3/PublicHolidays/{year}/BR'
        response = requests.get(url)
        response.raise_for_status()
        year_data = response.json()
        records.extend(year_data)

        add_elt_log(
            step='extract',
            object_name=f'Nager.Date API {year}',
            rows=len(year_data),
            cols=len(year_data[0].keys()) if len(year_data) > 0 else 0,
            notes=f'API endpoint: {url}'
        )

    holiday_raw = pd.DataFrame(records)

    raw_holiday_file = f'{RAW_HOLIDAY_PATH}/brazil_holidays_2016_2018_elt_raw.csv'
    holiday_raw.to_csv(raw_holiday_file, index=False)

    execution_time = round(time.time() - start_time, 3)
    add_elt_log(
        step='extract_summary',
        object_name='Nager.Date API raw extraction',
        rows=holiday_raw.shape[0],
        cols=holiday_raw.shape[1],
        execution_time_sec=execution_time,
        notes=f'Raw API data saved to {raw_holiday_file}'
    )

    return holiday_raw

holiday_raw = extract_elt_source2()
holiday_raw.head()

,date,localName,name,countryCode,fixed,global,counties,launchYear,types
0,2016-01-01,Confraternização Universal,New Year's Day,BR,False,True,None,None,[Public]
1,2016-02-08,Carnaval,Carnival,BR,False,True,None,None,"[Bank, Optional]"
2,2016-02-09,Carnaval,Carnival,BR,False,True,None,None,"[Bank, Optional]"
3,2016-03-25,Sexta-feira Santa,Good Friday,BR,False,True,None,None,[Public]
4,2016-03-27,Domingo de Páscoa,Easter Sunday,BR,False,True,None,None,[Public]


## 5. Load Raw Data Langsung ke Warehouse

In [ ]:
db_path = f'{WAREHOUSE_PATH}/bigdata_warehouse_elt.db'
conn = sqlite3.connect(db_path)

start_load = time.time()

raw_table_map = {
    'customers': 'raw_customers',
    'geolocation': 'raw_geolocation',
    'order_items': 'raw_order_items',
    'payments': 'raw_payments',
    'reviews': 'raw_reviews',
    'orders': 'raw_orders',
    'products': 'raw_products',
    'sellers': 'raw_sellers',
    'translation': 'raw_category_translation'
}

for source_name, table_name in raw_table_map.items():
    df = olist_raw[source_name]
    df.to_sql(table_name, conn, if_exists='replace', index=False)

    file_name = None
    for key, value in {
        'customers': 'olist_customers_dataset.csv',
        'geolocation': 'olist_geolocation_dataset.csv',
        'order_items': 'olist_order_items_dataset.csv',
        'payments': 'olist_order_payments_dataset.csv',
        'reviews': 'olist_order_reviews_dataset.csv',
        'orders': 'olist_orders_dataset.csv',
        'products': 'olist_products_dataset.csv',
        'sellers': 'olist_sellers_dataset.csv',
        'translation': 'product_category_name_translation.csv'
    }.items():
        if key == source_name:
            file_name = value
            break

    file_path = f'{RAW_OLIST_PATH}/{file_name}'
    add_metadata(source_name, table_name, df.shape[0], df.shape[1], get_file_size_mb(file_path), 'Loaded raw CSV directly to warehouse')

# Holiday raw: ubah list menjadi string agar aman masuk SQLite
holiday_raw_sql = holiday_raw.copy()
for col in holiday_raw_sql.columns:
    holiday_raw_sql[col] = holiday_raw_sql[col].apply(lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x)

holiday_raw_sql.to_sql('raw_holidays', conn, if_exists='replace', index=False)
add_metadata('Nager.Date Holiday API', 'raw_holidays', holiday_raw_sql.shape[0], holiday_raw_sql.shape[1], None, 'Loaded raw API JSON response to warehouse')

load_time = round(time.time() - start_load, 3)
add_elt_log('load_raw_summary', 'Raw data loaded into SQLite warehouse', execution_time_sec=load_time, notes=f'Database: {db_path}')

print('Raw data berhasil dimuat ke warehouse:', db_path)

Raw data berhasil dimuat ke warehouse: /content/drive/MyDrive/bigdata_final_project/warehouse/bigdata_warehouse_elt.db


## 6. Verifikasi Raw Tables dan Metadata Load

In [ ]:
raw_tables = list(raw_table_map.values()) + ['raw_holidays']

for table in raw_tables:
    result = pd.read_sql(f'SELECT COUNT(*) AS row_count FROM {table}', conn)
    print(table, result['row_count'].iloc[0])

metadata_df = pd.DataFrame(load_metadata)
metadata_df

raw_customers 99441
raw_geolocation 1000163
raw_order_items 112650
raw_payments 103886
raw_reviews 99224
raw_orders 99441
raw_products 32951
raw_sellers 3095
raw_category_translation 71
raw_holidays 42


,timestamp,source_name,target_table,rows,cols,file_size_mb,notes
0,2026-06-10 08:39:10,customers,raw_customers,99441,5,8.615,Loaded raw CSV directly to warehouse
1,2026-06-10 08:39:26,geolocation,raw_geolocation,1000163,5,58.435,Loaded raw CSV directly to warehouse
2,2026-06-10 08:39:32,order_items,raw_order_items,112650,7,14.723,Loaded raw CSV directly to warehouse
3,2026-06-10 08:39:34,payments,raw_payments,103886,5,5.510,Loaded raw CSV directly to warehouse
4,2026-06-10 08:39:39,reviews,raw_reviews,99224,7,13.782,Loaded raw CSV directly to warehouse
5,2026-06-10 08:39:45,orders,raw_orders,99441,8,16.837,Loaded raw CSV directly to warehouse
6,2026-06-10 08:39:45,products,raw_products,32951,9,2.269,Loaded raw CSV directly to warehouse
7,2026-06-10 08:39:45,sellers,raw_sellers,3095,4,0.167,Loaded raw CSV directly to warehouse
8,2026-06-10 08:39:45,translation,raw_category_translation,71,2,0.002,Loaded raw CSV directly to warehouse
9,2026-06-10 08:39:45,Nager.Date Holiday API,raw_holidays,42,9,NaN,Loaded raw API JSON response to warehouse


## 7. Transform ELT di Warehouse — Staging Tables Menggunakan SQL

In [ ]:
start_transform = time.time()
cursor = conn.cursor()

# Drop existing staging and final tables if rerun
objects_to_drop = [
    'stg_payments_agg', 'stg_reviews_agg', 'stg_geolocation_agg', 'stg_products_en', 'stg_fact_order_items_enriched',
    'dim_customer_elt', 'dim_product_elt', 'dim_seller_elt', 'dim_payment_elt', 'dim_date_elt', 'fact_order_items_elt'
]

for obj in objects_to_drop:
    cursor.execute(f'DROP TABLE IF EXISTS {obj};')

# Aggregated payments per order_id
cursor.execute('''
CREATE TABLE stg_payments_agg AS
SELECT
    order_id,
    SUM(payment_value) AS payment_value,
    SUM(payment_installments) AS payment_installments,
    MIN(payment_type) AS payment_type,
    COUNT(payment_sequential) AS payment_count
FROM raw_payments
GROUP BY order_id;
''')

# Aggregated reviews per order_id
cursor.execute('''
CREATE TABLE stg_reviews_agg AS
SELECT
    order_id,
    AVG(review_score) AS review_score,
    COUNT(review_id) AS review_count
FROM raw_reviews
GROUP BY order_id;
''')

# Aggregated geolocation per zip prefix
cursor.execute('''
CREATE TABLE stg_geolocation_agg AS
SELECT
    geolocation_zip_code_prefix,
    AVG(geolocation_lat) AS geolocation_lat,
    AVG(geolocation_lng) AS geolocation_lng,
    MIN(geolocation_city) AS geolocation_city,
    MIN(geolocation_state) AS geolocation_state
FROM raw_geolocation
GROUP BY geolocation_zip_code_prefix;
''')

# Products with English category
cursor.execute('''
CREATE TABLE stg_products_en AS
SELECT
    p.*,
    COALESCE(t.product_category_name_english, 'Unknown') AS product_category_name_english
FROM raw_products p
LEFT JOIN raw_category_translation t
    ON p.product_category_name = t.product_category_name;
''')

conn.commit()

add_elt_log('transform_staging', 'Staging aggregate tables created', notes='Payments, reviews, geolocation, and products translation transformed inside warehouse')
print('Staging tables berhasil dibuat.')

Staging tables berhasil dibuat.


## 8. Transform ELT — Enriched Fact Staging Menggunakan SQL

In [ ]:
cursor.execute('''
CREATE TABLE stg_fact_order_items_enriched AS
SELECT
    oi.order_id || '_' || CAST(oi.order_item_id AS TEXT) AS order_item_key,
    oi.order_id,
    oi.order_item_id,
    oi.product_id,
    oi.seller_id,
    oi.shipping_limit_date,
    oi.price,
    oi.freight_value,

    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,

    c.customer_unique_id,
    c.customer_zip_code_prefix,
    c.customer_city,
    c.customer_state,

    pa.payment_value,
    pa.payment_installments,
    pa.payment_type,
    pa.payment_count,

    p.product_category_name,
    p.product_category_name_english,
    p.product_name_lenght,
    p.product_description_lenght,
    p.product_photos_qty,
    p.product_weight_g,
    p.product_length_cm,
    p.product_height_cm,
    p.product_width_cm,

    s.seller_zip_code_prefix,
    s.seller_city,
    s.seller_state,

    r.review_score,
    r.review_count,

    g.geolocation_lat,
    g.geolocation_lng,
    g.geolocation_city,
    g.geolocation_state,

    DATE(o.order_purchase_timestamp) AS order_date,
    CAST(strftime('%Y', o.order_purchase_timestamp) AS INTEGER) AS order_year,
    CAST(strftime('%m', o.order_purchase_timestamp) AS INTEGER) AS order_month,
    CAST(strftime('%d', o.order_purchase_timestamp) AS INTEGER) AS order_day,
    CAST(strftime('%H', o.order_purchase_timestamp) AS INTEGER) AS order_hour,
    CASE
        WHEN strftime('%w', o.order_purchase_timestamp) IN ('0', '6') THEN 1
        ELSE 0
    END AS is_weekend,

    CAST(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp) AS INTEGER) AS delivery_days,
    CAST(julianday(o.order_estimated_delivery_date) - julianday(o.order_purchase_timestamp) AS INTEGER) AS estimated_delivery_days,
    CAST(julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) AS INTEGER) AS delivery_delay_days,
    CASE
        WHEN julianday(o.order_delivered_customer_date) - julianday(o.order_estimated_delivery_date) > 0 THEN 1
        ELSE 0
    END AS is_late_delivery,

    oi.price + oi.freight_value AS total_item_value,
    CASE
        WHEN oi.price > 0 THEN oi.freight_value / oi.price
        ELSE NULL
    END AS freight_ratio,

    p.product_length_cm * p.product_height_cm * p.product_width_cm AS product_volume_cm3,

    h.name AS holiday_name,
    h.localName AS holiday_local_name,
    CASE
        WHEN h.date IS NOT NULL THEN 1
        ELSE 0
    END AS is_public_holiday

FROM raw_order_items oi
LEFT JOIN raw_orders o ON oi.order_id = o.order_id
LEFT JOIN raw_customers c ON o.customer_id = c.customer_id
LEFT JOIN stg_payments_agg pa ON oi.order_id = pa.order_id
LEFT JOIN stg_products_en p ON oi.product_id = p.product_id
LEFT JOIN raw_sellers s ON oi.seller_id = s.seller_id
LEFT JOIN stg_reviews_agg r ON oi.order_id = r.order_id
LEFT JOIN stg_geolocation_agg g ON c.customer_zip_code_prefix = g.geolocation_zip_code_prefix
LEFT JOIN raw_holidays h ON DATE(o.order_purchase_timestamp) = h.date;
''')

conn.commit()

stg_shape = pd.read_sql('SELECT COUNT(*) AS rows FROM stg_fact_order_items_enriched;', conn)
print('stg_fact_order_items_enriched rows:', stg_shape['rows'].iloc[0])
add_elt_log('transform_join', 'stg_fact_order_items_enriched', rows=int(stg_shape['rows'].iloc[0]), notes='Warehouse-side SQL joins completed')

stg_fact_order_items_enriched rows: 112650


## 9. Transform ELT — Cleaning, Feature Final, dan Star Schema Menggunakan SQL

In [ ]:
# Drop final star schema tables jika cell ini dijalankan ulang
final_tables_to_drop = [
    'dim_customer_elt',
    'dim_product_elt',
    'dim_seller_elt',
    'dim_payment_elt',
    'dim_date_elt',
    'fact_order_items_elt'
]

for table in final_tables_to_drop:
    cursor.execute(f"DROP TABLE IF EXISTS {table};")

# Create dim_customer_elt
cursor.execute("""
CREATE TABLE dim_customer_elt AS
SELECT DISTINCT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix,
    COALESCE(customer_city, 'Unknown') AS customer_city,
    COALESCE(customer_state, 'Unknown') AS customer_state,
    geolocation_lat,
    geolocation_lng
FROM stg_fact_order_items_enriched
WHERE customer_id IS NOT NULL;
""")

# Create dim_product_elt
cursor.execute("""
CREATE TABLE dim_product_elt AS
SELECT DISTINCT
    product_id,
    COALESCE(product_category_name, 'Unknown') AS product_category_name,
    COALESCE(product_category_name_english, 'Unknown') AS product_category_name_english,
    COALESCE(product_name_lenght, 0) AS product_name_lenght,
    COALESCE(product_description_lenght, 0) AS product_description_lenght,
    COALESCE(product_photos_qty, 0) AS product_photos_qty,
    COALESCE(product_weight_g, 0) AS product_weight_g,
    COALESCE(product_length_cm, 0) AS product_length_cm,
    COALESCE(product_height_cm, 0) AS product_height_cm,
    COALESCE(product_width_cm, 0) AS product_width_cm,
    COALESCE(product_volume_cm3, 0) AS product_volume_cm3
FROM stg_fact_order_items_enriched
WHERE product_id IS NOT NULL;
""")

# Create dim_seller_elt
cursor.execute("""
CREATE TABLE dim_seller_elt AS
SELECT DISTINCT
    seller_id,
    seller_zip_code_prefix,
    COALESCE(seller_city, 'Unknown') AS seller_city,
    COALESCE(seller_state, 'Unknown') AS seller_state
FROM stg_fact_order_items_enriched
WHERE seller_id IS NOT NULL;
""")

# Create dim_payment_elt dengan surrogate key
cursor.execute("""
CREATE TABLE dim_payment_elt AS
SELECT
    ROW_NUMBER() OVER (
        ORDER BY
            COALESCE(payment_type, 'Unknown'),
            COALESCE(payment_installments, 0),
            COALESCE(payment_count, 0)
    ) AS payment_key,
    COALESCE(payment_type, 'Unknown') AS payment_type,
    COALESCE(payment_installments, 0) AS payment_installments,
    COALESCE(payment_count, 0) AS payment_count
FROM (
    SELECT DISTINCT
        payment_type,
        payment_installments,
        payment_count
    FROM stg_fact_order_items_enriched
);
""")

# FIXED: Create dim_date_elt dengan 1 baris per date_key
# Tidak memakai order_hour agar date_key unik.
cursor.execute("""
CREATE TABLE dim_date_elt AS
SELECT
    CAST(strftime('%Y%m%d', order_date) AS INTEGER) AS date_key,
    order_date AS date,
    MIN(order_year) AS order_year,
    MIN(order_month) AS order_month,
    MIN(order_day) AS order_day,
    MAX(is_weekend) AS is_weekend,
    MAX(is_public_holiday) AS is_public_holiday,
    MAX(COALESCE(holiday_name, 'Unknown')) AS holiday_name
FROM stg_fact_order_items_enriched
WHERE order_date IS NOT NULL
GROUP BY order_date;
""")

# Create fact_order_items_elt
cursor.execute("""
CREATE TABLE fact_order_items_elt AS
SELECT
    e.order_item_key,
    e.order_id,
    e.order_item_id,
    e.customer_id,
    e.product_id,
    e.seller_id,
    dp.payment_key,
    CAST(strftime('%Y%m%d', e.order_date) AS INTEGER) AS date_key,

    COALESCE(e.price, 0) AS price,
    COALESCE(e.freight_value, 0) AS freight_value,
    COALESCE(e.payment_value, 0) AS payment_value,
    COALESCE(e.review_score, 0) AS review_score,

    COALESCE(e.delivery_days, 0) AS delivery_days,
    COALESCE(e.estimated_delivery_days, 0) AS estimated_delivery_days,
    COALESCE(e.delivery_delay_days, 0) AS delivery_delay_days,
    COALESCE(e.is_late_delivery, 0) AS is_late_delivery,
    COALESCE(e.total_item_value, 0) AS total_item_value,
    COALESCE(e.freight_ratio, 0) AS freight_ratio,
    COALESCE(e.product_volume_cm3, 0) AS product_volume_cm3,
    COALESCE(e.is_public_holiday, 0) AS is_public_holiday
FROM stg_fact_order_items_enriched e
LEFT JOIN dim_payment_elt dp
    ON COALESCE(e.payment_type, 'Unknown') = dp.payment_type
    AND COALESCE(e.payment_installments, 0) = dp.payment_installments
    AND COALESCE(e.payment_count, 0) = dp.payment_count
WHERE e.order_item_key IS NOT NULL;
""")

conn.commit()

transform_time = round(time.time() - start_transform, 3)

add_elt_log(
    'transform_summary',
    'ELT star schema created inside warehouse',
    execution_time_sec=transform_time,
    notes='Fixed dim_date_elt with unique date_key'
)

print('Star schema ELT berhasil dibuat dengan dim_date_elt yang sudah fix.')

# Cek dim_date_elt apakah date_key sudah unik
check_dim_date = pd.read_sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT date_key) AS unique_date_key
FROM dim_date_elt;
""", conn)

display(check_dim_date)

Star schema ELT berhasil dibuat dengan dim_date_elt yang sudah fix.


,total_rows,unique_date_key
0,616,616


## 10. Dokumentasi Schema ELT dan Index

In [ ]:
schema_elt_sql = '''
-- Star Schema ELT Olist Data Warehouse
-- Raw data loaded first, then transformed inside SQLite warehouse.

-- Primary Key Documentation:
-- dim_customer_elt.customer_id
-- dim_product_elt.product_id
-- dim_seller_elt.seller_id
-- dim_payment_elt.payment_key
-- dim_date_elt.date_key
-- fact_order_items_elt.order_item_key

-- Foreign Key Documentation:
-- fact_order_items_elt.customer_id -> dim_customer_elt.customer_id
-- fact_order_items_elt.product_id -> dim_product_elt.product_id
-- fact_order_items_elt.seller_id -> dim_seller_elt.seller_id
-- fact_order_items_elt.payment_key -> dim_payment_elt.payment_key
-- fact_order_items_elt.date_key -> dim_date_elt.date_key
'''

schema_path = f'{WAREHOUSE_PATH}/schema_elt.sql'
with open(schema_path, 'w') as f:
    f.write(schema_elt_sql)

index_queries = [
    'CREATE INDEX IF NOT EXISTS idx_fact_elt_order_id ON fact_order_items_elt(order_id);',
    'CREATE INDEX IF NOT EXISTS idx_fact_elt_customer_id ON fact_order_items_elt(customer_id);',
    'CREATE INDEX IF NOT EXISTS idx_fact_elt_product_id ON fact_order_items_elt(product_id);',
    'CREATE INDEX IF NOT EXISTS idx_fact_elt_seller_id ON fact_order_items_elt(seller_id);',
    'CREATE INDEX IF NOT EXISTS idx_fact_elt_date_key ON fact_order_items_elt(date_key);'
]

for q in index_queries:
    cursor.execute(q)

conn.commit()
print('schema_elt.sql dan index berhasil dibuat:', schema_path)

schema_elt.sql dan index berhasil dibuat: /content/drive/MyDrive/bigdata_final_project/warehouse/schema_elt.sql


## 11. Verifikasi Tabel Star Schema ELT

In [ ]:
final_tables = [
    'dim_customer_elt',
    'dim_product_elt',
    'dim_seller_elt',
    'dim_payment_elt',
    'dim_date_elt',
    'fact_order_items_elt'
]

for table in final_tables:
    count_df = pd.read_sql(f'SELECT COUNT(*) AS row_count FROM {table}', conn)
    print(table, count_df['row_count'].iloc[0])

dim_customer_elt 98666
dim_product_elt 32951
dim_seller_elt 3095
dim_payment_elt 107
dim_date_elt 616
fact_order_items_elt 112650


## 12. Validasi Kualitas Data ELT

In [ ]:
validation_queries = {
    'uniqueness_check_order_item_key': '''
        SELECT COUNT(*) - COUNT(DISTINCT order_item_key) AS failed_count
        FROM fact_order_items_elt;
    ''',
    'null_check_primary_keys': '''
        SELECT COUNT(*) AS failed_count
        FROM fact_order_items_elt
        WHERE order_item_key IS NULL OR order_id IS NULL OR product_id IS NULL OR seller_id IS NULL;
    ''',
    'range_check_price_freight_non_negative': '''
        SELECT COUNT(*) AS failed_count
        FROM fact_order_items_elt
        WHERE price < 0 OR freight_value < 0;
    ''',
    'referential_integrity_product_id': '''
        SELECT COUNT(*) AS failed_count
        FROM fact_order_items_elt f
        LEFT JOIN dim_product_elt p ON f.product_id = p.product_id
        WHERE p.product_id IS NULL;
    ''',
    'row_count_minimum_100k': '''
        SELECT CASE WHEN COUNT(*) >= 100000 THEN 0 ELSE 1 END AS failed_count
        FROM fact_order_items_elt;
    ''',
    'column_count_minimum_12': '''
        SELECT CASE WHEN COUNT(*) >= 12 THEN 0 ELSE 1 END AS failed_count
        FROM pragma_table_info('fact_order_items_elt');
    '''
}

validation_results = []

for rule_name, query in validation_queries.items():
    result = pd.read_sql(query, conn)
    failed_count = int(result['failed_count'].iloc[0])
    validation_results.append({
        'rule_name': rule_name,
        'passed': failed_count == 0,
        'failed_count': failed_count
    })

validation_elt_df = pd.DataFrame(validation_results)
validation_elt_path = f'{WAREHOUSE_PATH}/elt_validation_results.csv'
validation_elt_df.to_csv(validation_elt_path, index=False)

validation_elt_df

,rule_name,passed,failed_count
0,uniqueness_check_order_item_key,True,0
1,null_check_primary_keys,True,0
2,range_check_price_freight_non_negative,True,0
3,referential_integrity_product_id,True,0
4,row_count_minimum_100k,True,0
5,column_count_minimum_12,True,0


## 13. Query Analitik ELT untuk Verifikasi dan Eksplorasi

In [ ]:
elt_analytic_queries = {}

elt_analytic_queries['q1_monthly_revenue_elt'] = '''
SELECT
    d.order_year,
    d.order_month,
    COUNT(DISTINCT f.order_id) AS total_orders,
    COUNT(*) AS total_items,
    ROUND(SUM(f.total_item_value), 2) AS total_revenue
FROM fact_order_items_elt f
JOIN dim_date_elt d ON f.date_key = d.date_key
GROUP BY d.order_year, d.order_month
ORDER BY d.order_year, d.order_month;
'''

elt_analytic_queries['q2_category_revenue_elt'] = '''
SELECT
    p.product_category_name_english,
    COUNT(*) AS total_items,
    ROUND(SUM(f.total_item_value), 2) AS total_revenue,
    ROUND(AVG(f.review_score), 2) AS avg_review_score
FROM fact_order_items_elt f
JOIN dim_product_elt p ON f.product_id = p.product_id
GROUP BY p.product_category_name_english
ORDER BY total_revenue DESC
LIMIT 10;
'''

elt_analytic_queries['q3_payment_type_elt'] = '''
SELECT
    p.payment_type,
    COUNT(*) AS total_items,
    COUNT(DISTINCT f.order_id) AS total_orders,
    ROUND(SUM(f.payment_value), 2) AS total_payment_value
FROM fact_order_items_elt f
JOIN dim_payment_elt p ON f.payment_key = p.payment_key
GROUP BY p.payment_type
ORDER BY total_payment_value DESC;
'''

elt_analytic_queries['q4_customer_state_revenue_elt'] = '''
SELECT
    c.customer_state,
    COUNT(DISTINCT f.order_id) AS total_orders,
    COUNT(*) AS total_items,
    ROUND(SUM(f.total_item_value), 2) AS total_revenue
FROM fact_order_items_elt f
JOIN dim_customer_elt c ON f.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY total_revenue DESC
LIMIT 10;
'''

elt_analytic_queries['q5_late_delivery_review_elt'] = '''
SELECT
    is_late_delivery,
    COUNT(*) AS total_items,
    ROUND(AVG(delivery_delay_days), 2) AS avg_delivery_delay_days,
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM fact_order_items_elt
GROUP BY is_late_delivery;
'''

elt_analytic_queries['q6_holiday_nonholiday_elt'] = '''
SELECT
    d.is_public_holiday,
    COUNT(DISTINCT f.order_id) AS total_orders,
    COUNT(*) AS total_items,
    ROUND(SUM(f.total_item_value), 2) AS total_revenue,
    ROUND(AVG(f.review_score), 2) AS avg_review_score
FROM fact_order_items_elt f
JOIN dim_date_elt d ON f.date_key = d.date_key
GROUP BY d.is_public_holiday;
'''

queries_path = f"{WAREHOUSE_PATH}/elt_analytic_queries.sql"

with open(queries_path, "w", encoding="utf-8") as f:
    for name, query in elt_analytic_queries.items():
        f.write("-- " + name + "\\n")
        f.write(query.strip() + "\\n\\n")

print("Query analitik ELT tersimpan di:", queries_path)

for name, query in elt_analytic_queries.items():
    print("\\n==============================")
    print(name)
    print("==============================")
    display(pd.read_sql(query, conn).head(10))

Query analitik ELT tersimpan di: /content/drive/MyDrive/bigdata_final_project/warehouse/elt_analytic_queries.sql
\n==============================
q1_monthly_revenue_elt


,order_year,order_month,total_orders,total_items,total_revenue
0,2016,9,3,6,354.75
1,2016,10,308,363,56808.84
2,2016,12,1,1,19.62
3,2017,1,789,955,137188.49
4,2017,2,1733,1951,286280.62
5,2017,3,2641,3000,432048.59
6,2017,4,2391,2684,412422.24
7,2017,5,3660,4136,586190.95
8,2017,6,3217,3583,502963.04
9,2017,7,3969,4519,584971.62


\n==============================
q2_category_revenue_elt


,product_category_name_english,total_items,total_revenue,avg_review_score
0,health_beauty,9670,1441248.07,4.11
1,watches_gifts,5991,1305541.61,3.98
2,bed_bath_table,11115,1241681.72,3.85
3,sports_leisure,8641,1156656.48,4.08
4,computers_accessories,7827,1059272.40,3.91
5,furniture_decor,8334,902511.79,3.87
6,housewares,6964,778397.77,4.03
7,cool_stuff,3796,719329.95,4.11
8,auto,4235,685384.32,4.02
9,garden_tools,4347,584219.21,4.02


\n==============================
q3_payment_type_elt


...

,customer_state,total_orders,total_items,total_revenue
0,SP,41375,47449,5921678.12
1,RJ,12762,14579,2129681.98
2,MG,11544,13129,1856161.49
3,RS,5432,6235,885826.76
4,PR,4998,5740,800935.44
5,BA,3358,3799,611506.67
6,SC,3612,4176,610213.60
7,DF,2125,2406,353229.44
8,GO,2007,2333,347706.93
9,ES,2025,2256,324801.91


\n==============================
q5_late_delivery_review_elt


,is_late_delivery,total_items,avg_delivery_delay_days,avg_review_score
0,0,103935,-12.51,4.13
1,1,8715,8.74,2.49


\n==============================
q6_holiday_nonholiday_elt


,is_public_holiday,total_orders,total_items,total_revenue,avg_review_score
0,0,95878,109507,15415833.48,4.00
1,1,2788,3143,427719.76,3.93


## 14. Simpan Log dan Metadata ELT

In [ ]:
log_df, metadata_df = save_elt_logs()

# Simpan metadata dan log juga ke warehouse agar dokumentasi load tersedia di database
log_df.to_sql('metadata_elt_process_log', conn, if_exists='replace', index=False)
metadata_df.to_sql('metadata_elt_load_metadata', conn, if_exists='replace', index=False)

print('Metadata log juga disimpan ke SQLite warehouse.')
display(log_df)
display(metadata_df)

Log ELT tersimpan di: /content/drive/MyDrive/bigdata_final_project/elt_pipeline/logs/elt_process_log.csv
Metadata load ELT tersimpan di: /content/drive/MyDrive/bigdata_final_project/elt_pipeline/logs/elt_load_metadata.csv
Metadata log juga disimpan ke SQLite warehouse.


,timestamp,step,object_name,rows,cols,execution_time_sec,status,notes
0,2026-06-10 08:38:49,extract,raw_csv_customers,99441.0,5.0,NaN,success,Extract raw CSV from /content/drive/MyDrive/bi...
1,2026-06-10 08:38:53,extract,raw_csv_geolocation,1000163.0,5.0,NaN,success,Extract raw CSV from /content/drive/MyDrive/bi...
2,2026-06-10 08:38:54,extract,raw_csv_order_items,112650.0,7.0,NaN,success,Extract raw CSV from /content/drive/MyDrive/bi...
3,2026-06-10 08:38:55,extract,raw_csv_payments,103886.0,5.0,NaN,success,Extract raw CSV from /content/drive/MyDrive/bi...
4,2026-06-10 08:38:57,extract,raw_csv_reviews,99224.0,7.0,NaN,success,Extract raw CSV from /content/drive/MyDrive/bi...
5,2026-06-10 08:38:58,extract,raw_csv_orders,99441.0,8.0,NaN,success,Extract raw CSV from /content/drive/MyDrive/bi...
6,2026-06-10 08:38:58,extract,raw_csv_products,32951.0,9.0,NaN,success,Extract raw CSV from /content/drive/MyDrive/bi...
7,2026-06-10 08:38:59,extract,raw_csv_sellers,3095.0,4.0,NaN,success,Extract raw CSV from /content/drive/MyDrive/bi...
8,2026-06-10 08:38:59,extract,raw_csv_translation,71.0,2.0,NaN,success,Extract raw CSV from /content/drive/MyDrive/bi...
9,2026-06-10 08:38:59,extract_summary,Olist CSV raw extraction,NaN,NaN,11.339,success,


,timestamp,source_name,target_table,rows,cols,file_size_mb,notes
0,2026-06-10 08:39:10,customers,raw_customers,99441,5,8.615,Loaded raw CSV directly to warehouse
1,2026-06-10 08:39:26,geolocation,raw_geolocation,1000163,5,58.435,Loaded raw CSV directly to warehouse
2,2026-06-10 08:39:32,order_items,raw_order_items,112650,7,14.723,Loaded raw CSV directly to warehouse
3,2026-06-10 08:39:34,payments,raw_payments,103886,5,5.510,Loaded raw CSV directly to warehouse
4,2026-06-10 08:39:39,reviews,raw_reviews,99224,7,13.782,Loaded raw CSV directly to warehouse
5,2026-06-10 08:39:45,orders,raw_orders,99441,8,16.837,Loaded raw CSV directly to warehouse
6,2026-06-10 08:39:45,products,raw_products,32951,9,2.269,Loaded raw CSV directly to warehouse
7,2026-06-10 08:39:45,sellers,raw_sellers,3095,4,0.167,Loaded raw CSV directly to warehouse
8,2026-06-10 08:39:45,translation,raw_category_translation,71,2,0.002,Loaded raw CSV directly to warehouse
9,2026-06-10 08:39:45,Nager.Date Holiday API,raw_holidays,42,9,NaN,Loaded raw API JSON response to warehouse


## 15. Ringkasan Perbedaan Pendekatan ETL dan ELT

In [ ]:
comparison_summary = pd.DataFrame([
    {
        'aspect': 'Urutan proses',
        'etl_pipeline': 'Extract -> Transform with Pandas -> Load curated/star schema',
        'elt_pipeline': 'Extract -> Load raw tables -> Transform with SQL inside warehouse'
    },
    {
        'aspect': 'Lokasi transformasi',
        'etl_pipeline': 'Sebelum data masuk warehouse',
        'elt_pipeline': 'Setelah raw data masuk warehouse'
    },
    {
        'aspect': 'Output warehouse',
        'etl_pipeline': 'Warehouse ETL langsung berisi data hasil transformasi',
        'elt_pipeline': 'Warehouse ELT berisi raw tables, staging tables, dan final star schema'
    },
    {
        'aspect': 'Teknologi transformasi',
        'etl_pipeline': 'Pandas/Python',
        'elt_pipeline': 'SQL di SQLite warehouse'
    }
])

comparison_path = f'{WAREHOUSE_PATH}/etl_elt_comparison_summary.csv'
comparison_summary.to_csv(comparison_path, index=False)
comparison_summary

,aspect,etl_pipeline,elt_pipeline
0,Urutan proses,Extract -> Transform with Pandas -> Load curat...,Extract -> Load raw tables -> Transform with S...
1,Lokasi transformasi,Sebelum data masuk warehouse,Setelah raw data masuk warehouse
2,Output warehouse,Warehouse ETL langsung berisi data hasil trans...,"Warehouse ELT berisi raw tables, staging table..."
3,Teknologi transformasi,Pandas/Python,SQL di SQLite warehouse


## Catatan Final Notebook 2

Notebook ini memenuhi kebutuhan Bagian II – Pipeline ELT:

- Dataset mentah diambil dari sumber yang sama dengan ETL.
- Data mentah langsung dimuat ke raw tables di SQLite warehouse tanpa preprocessing.
- Metadata proses load didokumentasikan.
- Transformasi dilakukan setelah data berada di warehouse.
- Join antar tabel dilakukan di sisi warehouse menggunakan SQL.
- Agregasi, perhitungan metrik, dan feature engineering dilakukan dengan SQL.
- Pendekatan berbeda dari ETL karena transformasi utama tidak dilakukan dengan Pandas sebelum load, melainkan di warehouse setelah raw data dimuat.
